<a href="https://colab.research.google.com/github/AnDinh039/PST-PRIM---RMAT_128.edges---Julia/blob/main/WIN_SUM%20-%20VEC-512%20-%20JULIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <img src="https://github.com/JuliaLang/julia-logo-graphics/raw/master/images/julia-logo-color.png" height="100" /> _Colab Notebook Template_

## Instructions
1. Work on a copy of this notebook: _File_ > _Save a copy in Drive_ (you will need a Google account). Alternatively, you can download the notebook using _File_ > _Download .ipynb_, then upload it to [Colab](https://colab.research.google.com/).
2. If you need a GPU: _Runtime_ > _Change runtime type_ > _Harware accelerator_ = _GPU_.
3. Execute the following cell (click on it and press Ctrl+Enter) to install Julia, IJulia and other packages (if needed, update `JULIA_VERSION` and the other parameters). This takes a couple of minutes.
4. Reload this page (press Ctrl+R, or ⌘+R, or the F5 key) and continue to the next section.

_Notes_:
* If your Colab Runtime gets reset (e.g., due to inactivity), repeat steps 2, 3 and 4.
* After installation, if you want to change the Julia version or activate/deactivate the GPU, you will need to reset the Runtime: _Runtime_ > _Factory reset runtime_ and repeat steps 3 and 4.

In [1]:
%%shell
set -e

#---------------------------------------------------#
JULIA_VERSION="1.8.2" # any version ≥ 0.7.0
JULIA_PACKAGES="IJulia BenchmarkTools"
JULIA_PACKAGES_IF_GPU="CUDA" # or CuArrays for older Julia versions
JULIA_NUM_THREADS=2
#---------------------------------------------------#

if [ -z `which julia` ]; then
  # Install Julia
  JULIA_VER=`cut -d '.' -f -2 <<< "$JULIA_VERSION"`
  echo "Installing Julia $JULIA_VERSION on the current Colab Runtime..."
  BASE_URL="https://julialang-s3.julialang.org/bin/linux/x64"
  URL="$BASE_URL/$JULIA_VER/julia-$JULIA_VERSION-linux-x86_64.tar.gz"
  wget -nv $URL -O /tmp/julia.tar.gz # -nv means "not verbose"
  tar -x -f /tmp/julia.tar.gz -C /usr/local --strip-components 1
  rm /tmp/julia.tar.gz

  # Install Packages
  nvidia-smi -L &> /dev/null && export GPU=1 || export GPU=0
  if [ $GPU -eq 1 ]; then
    JULIA_PACKAGES="$JULIA_PACKAGES $JULIA_PACKAGES_IF_GPU"
  fi
  for PKG in `echo $JULIA_PACKAGES`; do
    echo "Installing Julia package $PKG..."
    julia -e 'using Pkg; pkg"add '$PKG'; precompile;"' &> /dev/null
  done

  # Install kernel and rename it to "julia"
  echo "Installing IJulia kernel..."
  julia -e 'using IJulia; IJulia.installkernel("julia", env=Dict(
      "JULIA_NUM_THREADS"=>"'"$JULIA_NUM_THREADS"'"))'
  KERNEL_DIR=`julia -e "using IJulia; print(IJulia.kerneldir())"`
  KERNEL_NAME=`ls -d "$KERNEL_DIR"/julia*`
  mv -f $KERNEL_NAME "$KERNEL_DIR"/julia

  echo ''
  echo "Successfully installed `julia -v`!"
  echo "Please reload this page (press Ctrl+R, ⌘+R, or the F5 key) then"
  echo "jump to the 'Checking the Installation' section."
fi

Unrecognized magic `%%shell`.

Julia does not use the IPython `%magic` syntax.   To interact with the IJulia kernel, use `IJulia.somefunction(...)`, for example.  Julia macros, string macros, and functions can be used to accomplish most of the other functionalities of IPython magics.


# Checking the Installation
The `versioninfo()` function should print your Julia version and some other info about the system:

In [2]:
versioninfo()

Julia Version 1.12.6
Commit 15346901f00 (2026-04-09 19:20 UTC)
Build Info:
  Official https://julialang.org release
Platform Info:
  OS: Linux (x86_64-linux-gnu)
  CPU: 2 × Intel(R) Xeon(R) CPU @ 2.20GHz
  WORD_SIZE: 64
  LLVM: libLLVM-18.1.7 (ORCJIT, broadwell)
  GC: Built with stock GC
Threads: 2 default, 1 interactive, 2 GC (on 2 virtual cores)
Environment:
  JULIA_NUM_THREADS = auto


In [3]:
import Pkg

In [4]:
Pkg.add("BenchmarkTools")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed BenchmarkTools ─ v1.8.0
    Updating `~/.julia/environments/v1.12/Project.toml`
  [6e4b80f9] + BenchmarkTools v1.8.0
    Updating `~/.julia/environments/v1.12/Manifest.toml`
  [6e4b80f9] + BenchmarkTools v1.8.0
  [9abbd945] + Profile v1.11.0
Precompiling packages...
   1196.0 ms  ✓ RangeArrays
   2631.1 ms  ✓ ExpressionExplorer
   1568.0 ms  ✓ AbstractFFTs
   1055.3 ms  ✓ LaTeXStrings
   1947.0 ms  ✓ DataStrings
   1045.6 ms  ✓ IndirectArrays
   1142.8 ms  ✓ PolygonOps
   1515.5 ms  ✓ ExprTools
    986.1 ms  ✓ IteratorInterfaceExtensions
   1400.7 ms  ✓ TensorCore
   2367.0 ms  ✓ StatsAPI
   1999.2 ms  ✓ Contour
   1657.2 ms  ✓ HelpPlots
  11151.4 ms  ✓ UnsafeAtomics
   5131.2 ms  ✓ CEnum
   3801.3 ms  ✓ OffsetArrays
   1550.6 ms  ✓ TriplotBase
  22613.5 ms  ✓ BaseDirs
   2377.2 ms  ✓ IntegerMathUtils
   5411.9 ms  ✓ FillArrays
   6271.5 ms  ✓ Format
   1997.9 ms  ✓ Measures
   141

LoadError: InterruptException:

In [6]:
try
    using CUDA
catch
    println("No GPU found.")
else
    run(`nvidia-smi`)
    # Create a new random matrix directly on the GPU:
    M_on_gpu = CUDA.CURAND.rand(2^11, 2^11)
    @btime $M_on_gpu * $M_on_gpu; nothing
end

No GPU found.


In [17]:
W    = 30
path = "vec.csv"        # đổi nếu file nằm ở thư mục khác, ví dụ "/content/vec.csv"

# ---------- Đọc dataset ----------
lines = readlines(path)
rows  = [split(l, ",") for l in lines[2:end] if !isempty(strip(l))]   # bỏ dòng tiêu đề
n     = length(rows)
vecA  = zeros(Int, n)
for r in rows
    vecA[parse(Int, r[1])] = parse(Int, r[2])    # vecA[i] = a
end

# ---------- Thuật toán: TempoDL sau subsumption và temporal elimination ----------
function winsum_tempo(vecA, w, n)
    sp = 0                                   # winsum'(s) := (s = 0)
    for i in 0:n-1                           # for i from 0 ... end
        s = sp                               # winsum := winsum'
        if i < w
            sp = s + vecA[i+1]                           # luật (2), i < W
        else
            sp = s + vecA[i+1] - vecA[i-w+1]             # luật (3), i >= W
        end
    end
    return sp                                # query(s) := winsum'(s)
end

# ---------- Chạy đúng 1 lần ----------
t0 = time_ns()
kq = winsum_tempo(vecA, W, n)
t1 = time_ns()

println("n = $n")
println("winsum($n) = $kq")
println("thời gian = $((t1 - t0) / 1e9) giây")


n = 512
winsum(512) = 165
thời gian = 0.02859802 giây
